<a href="https://colab.research.google.com/github/AriesConsulting/808s-Auto-Generations-/blob/main/Spatial_Quantum_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import librosa
import tkinter as tk
from tkinter import filedialog

class SpatialQuantumEngine:
    def __init__(self):
        self.sun_pos = [0, 0]
        self.ghost_layer = None
        self.velocity_field = None  # For Temporal Feedback
        self.decay_rate = 0.92      # How long ripples last

    def apply_unified_distortion(self, img, sun_x, sun_y, loudness, chromatic_strength=0.22):
        """
        Unified remap operation combining Gravitational Pull, Chromatic Aberration,
        and Temporal Ripples.
        """
        h, w = img.shape[:2]

        # Initialize velocity field (Temporal Feedback) if not exists
        if self.velocity_field is None:
            self.velocity_field = np.zeros((h, w, 2), dtype=np.float32)

        # Create base coordinate grids
        x, y = np.meshgrid(np.arange(w), np.arange(h))

        # 1. Calculate Instantaneous Gravitational Pull
        dx = x - sun_x
        dy = y - sun_y
        dist = np.sqrt(dx**2 + dy**2) + 1e-6

        # Pull strength scales with loudness
        pull_mag = (loudness * 85) / (dist / 45 + 1)

        # Calculate current frame's force vectors
        current_force_x = (dx / dist) * pull_mag
        current_force_y = (dy / dist) * pull_mag

        # 2. Integrate Temporal Feedback (The Ripple Effect)
        # We add the new force to the existing field and apply decay
        self.velocity_field[:, :, 0] = (self.velocity_field[:, :, 0] + current_force_x * 0.2) * self.decay_rate
        self.velocity_field[:, :, 1] = (self.velocity_field[:, :, 1] + current_force_y * 0.2) * self.decay_rate

        # Apply the cumulative displacement field
        flex_x = x - self.velocity_field[:, :, 0]
        flex_y = y - self.velocity_field[:, :, 1]

        # 3. Add Chromatic Lens on top of warped coordinates
        # Normalize coordinates for radial calculation
        x_n = (2.0 * flex_x - w) / w
        y_n = (2.0 * flex_y - h) / h
        r_radial = np.sqrt(x_n**2 + y_n**2)

        output = []
        # Distinct shifts for RGB channels
        shifts = [1.08, 1.0, 0.92]
        for s_mult in shifts:
            s = chromatic_strength * s_mult
            dist_factor = 1.0 + s * (r_radial**2)

            # Map normalized warped coords back to pixel space
            mx = (((x_n * dist_factor) + 1.0) * w / 2.0).astype(np.float32)
            my = (((y_n * dist_factor) + 1.0) * h / 2.0).astype(np.float32)

            output.append(cv2.remap(img, mx, my, cv2.INTER_LINEAR))

        # Reconstruct RGB from the specifically distorted channels
        # Channel 0 of output[0] is Red, Channel 1 of output[1] is Green, etc.
        res = cv2.merge([
            output[0][:, :, 0],
            output[1][:, :, 1],
            output[2][:, :, 2]
        ])

        return res

    def process_frame(self, frame, loudness, pan_x, pan_y, freq_color):
        h, w, _ = frame.shape
        if self.ghost_layer is None:
            self.ghost_layer = np.zeros_like(frame)

        # 1. Update Quantum Sun Position (Stereo Panning)
        target_x = int((w/2) + (pan_x * w/3))
        target_y = int((h/2) - (pan_y * h/3))

        # 2. Render the "Sun" (Light Source)
        sun_radius = int(25 + (loudness * 80))
        sun_overlay = np.zeros_like(frame)
        cv2.circle(sun_overlay, (target_x, target_y), sun_radius, freq_color, -1)
        sun_overlay = cv2.GaussianBlur(sun_overlay, (75, 75), 0)

        # 3. Motion Blur / Ghosting Trail (Persistence of light)
        self.ghost_layer = cv2.addWeighted(self.ghost_layer, 0.88, sun_overlay, 0.20, 0)

        # 4. Glitch Base (Pre-warp RGB Shift for extra "shiver")
        b, g, r = cv2.split(frame)
        shift_amount = int(loudness * 30)
        r = np.roll(r, shift_amount, axis=1)
        b = np.roll(b, -shift_amount, axis=1)
        glitched = cv2.merge([b, g, r])

        # 5. Apply the Unified Gravity + Lens Warp + Temporal Feedback
        warped_base = self.apply_unified_distortion(glitched, target_x, target_y, loudness)

        # Merge Sun/Ghosting with the Warp
        combined_visual = cv2.add(warped_base, self.ghost_layer)

        # 6. Quantum Dot Grid (Spatial Reference)
        dot_canvas = np.zeros_like(frame)
        dot_res = 24
        for y_step in range(0, h, dot_res):
            for x_step in range(0, w, dot_res):
                # Dots also get "pulled" slightly for visual consistency
                d_to_sun = np.sqrt((x_step-target_x)**2 + (y_step-target_y)**2)
                dot_r = max(1, int(14 * (1 - d_to_sun/w) * loudness))
                cv2.circle(dot_canvas, (x_step, y_step), dot_r, freq_color, -1)

        return np.hstack((combined_visual, dot_canvas))

# --- Main Execution ---
if __name__ == "__main__":
    root = tk.Tk()
    root.withdraw()
    path = filedialog.askopenfilename(title="Select Video/Audio File")
    if not path: exit()

    print(f"Loading: {path}")
    y, sr = librosa.load(path, mono=False)

    if y.ndim > 1:
        left, right = y[0], y[1]
        rms_l = librosa.feature.rms(y=left)[0]
        rms_r = librosa.feature.rms(y=right)[0]
        pan_x = (rms_r - rms_l) / (rms_l + rms_r + 1e-6)
        rms_total = (rms_l + rms_r) / 2
    else:
        pan_x = np.zeros(100)
        rms_total = librosa.feature.rms(y=y)[0]

    # Normalize Loudness
    rms_total = (rms_total - np.min(rms_total)) / (np.max(rms_total) - np.min(rms_total) + 1e-6)

    engine = SpatialQuantumEngine()
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w_orig = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_orig = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Output Writer
    out = cv2.VideoWriter("quantum_feedback_output.mp4",
                         cv2.VideoWriter_fourcc(*'mp4v'),
                         fps, (w_orig * 2, h_orig))

    f_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        a_idx = int((f_idx / total_frames) * len(rms_total))
        cur_loud = rms_total[min(a_idx, len(rms_total)-1)]
        cur_pan = pan_x[min(a_idx, len(pan_x)-1)] if y.ndim > 1 else 0

        # Periodic vertical oscillation
        cur_pan_y = (np.sin(f_idx * 0.08) * 0.15) * cur_loud

        # Trigonometric Color Cycle (BGR format)
        color = (
            int(127 + 127 * np.sin(f_idx * 0.04)),
            int(127 + 127 * np.cos(f_idx * 0.02)),
            200
        )

        res = engine.process_frame(frame, cur_loud, cur_pan, cur_pan_y, color)
        out.write(res)

        # Display Preview
        cv2.imshow('Quantum Feedback Engine', cv2.resize(res, (1200, 500)))

        f_idx += 1
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("Processing Complete.")